# Для диабетика: умный поиск + сборка дня

Два режима:
1. **«Можно ли мне X?»** — вводишь название, получаешь 5 ближайших продуктов с меткой для диабета (🟢 рекомендовано / ✅ разрешено / ⚠️ осторожно / ⛔ запрещено).
2. **Сборка дня** — добавляешь продукты + граммы, видишь выполнение нормы и предупреждения по сахару/натрию.

Метка для диабета считается из гликемического индекса + сахара (см. `diet/gi.py`).
> ⚠️ Ориентировочно, не заменяет врача.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import pandas as pd
from diet import (UserProfile, calculate, search_for_diabetes, LABEL_ICONS,
                  FoodLog, check_day, to_fooditem)
pd.set_option('display.width', 200); pd.set_option('display.max_colwidth', 42)

FOODS = pd.read_csv('../data/processed/unified_foods.csv')
log = FoodLog()
print(f'База: {len(FOODS)} продуктов с метками для диабета')

База: 30965 продуктов с метками для диабета


## Часть 1. «Можно ли мне это?» — умный поиск

Введи название продукта. Получишь топ-5, отсортированных от самых подходящих к неподходящим для диабета.

In [15]:
# ↓↓↓ ВВЕДИ СВОЙ ЗАПРОС ↓↓↓
query = 'сосиска'   # попробуй: творог, рис, хлеб, конфета, яблоко, кола, картофель

res = search_for_diabetes(FOODS, query, limit=5)
print(f'Поиск «{query}» — топ-5 для диабета 2 типа:')
print()
for i, r in res.iterrows():
    icon = LABEL_ICONS.get(r['diabetes_label'], r['diabetes_label'])
    gi = r['gi']
    sug = r['sugars_g']
    sug_s = f'{sug:g}' if pd.notna(sug) else '?'
    print(f'  {i+1}. {str(r["name"])[:40]:<40} ГИ:{gi:>3}  сахар:{sug_s:>4}г  {icon}')
    print(f'     категория: {r["food_group"]}; причина: {r["diabetes_reason"]}')

Поиск «сосиска» — топ-5 для диабета 2 типа:

  1. Соус Моя семья к сосискам с горчичными с ГИ: 27  сахар:   ?г  🟢 рекомендовано
     категория: Орехи; причина: ГИ 27 ≤45 — низкий, рекомендуется
  2. сосиска ,,здоровая ,,                    ГИ:  0  сахар:   ?г  ✅ разрешено
     категория: Мясо/рыба/яйца (ГИ≈0); причина: мясо/рыба/яйца — без углеводов, безопасно
  3. сосиска ,,здорово ,,                     ГИ:  0  сахар:   ?г  ✅ разрешено
     категория: Мясо/рыба/яйца (ГИ≈0); причина: мясо/рыба/яйца — без углеводов, безопасно
  4. сосиска здоровая                         ГИ:  0  сахар:   ?г  ✅ разрешено
     категория: Мясо/рыба/яйца (ГИ≈0); причина: мясо/рыба/яйца — без углеводов, безопасно
  5. сосиска здоровая                         ГИ:  0  сахар:   ?г  ✅ разрешено
     категория: Мясо/рыба/яйца (ГИ≈0); причина: мясо/рыба/яйца — без углеводов, безопасно


## Часть 2. Сборка дня для диабетика

### 2.1 Профиль и норма дня

In [3]:
# ↓↓↓ ЗАПОЛНИ ПОД СЕБЯ ↓↓↓
profile_data = {
    "sex": "female", "age": 55, "weight": 82,
    "height": 165, "activity": "light", "goal": "lose",
}
target = calculate(UserProfile(**profile_data), condition='diabetes_t2', formula='who')
print(f'{target.condition_label} · {target.goal_label}')
print(f'Цель: {round(target.target_kcal)} ккал | Б{round(target.protein_g)} Ж{round(target.fat_g)} У{round(target.carbs_g)}')
print(f'Лимиты при диабете: сахар ≤30 г/день, натрий <2300 мг/день')

Сахарный диабет 2 типа · Снижение веса
Цель: 1614 ккал | Б73 Ж54 У210
Лимиты при диабете: сахар ≤30 г/день, натрий <2300 мг/день


### 2.2 Поиск продукта для добавления

Введи что хочешь добавить — увидишь варианты с метками и индексы строк.

In [14]:
# ↓↓↓ ЧТО ИЩЕМ ↓↓↓
query = 'сосиски'   # попробуй: курица, овсян, яйцо, творог, овощи

res = search_for_diabetes(FOODS, query, limit=8)
for i, r in res.iterrows():
    icon = LABEL_ICONS.get(r['diabetes_label'], r['diabetes_label'])
    kcal = r['kcal']
    kcal_s = f'{kcal:g}' if pd.notna(kcal) else '?'
    print(f'  [{i}] {str(r["name"])[:42]:<42} {kcal_s:>4}ккал  {icon}')

  [0] Сосиски Баварские оригинальные с сыром      319ккал  🟢 рекомендовано
  [1] Сосиски &quot;Вязанка Сливочные&quot;       190ккал  🟢 рекомендовано
  [2] Сосиски Зел.лин                               ?ккал  🟢 рекомендовано
  [3] Сосиски Клинские с сыром                      ?ккал  🟢 рекомендовано
  [4] Сосиски С Сыром                               ?ккал  🟢 рекомендовано
  [5] Сосиски Фестивальки, Калинка                  ?ккал  🟢 рекомендовано
  [6] Сосиски агрокомплекс с сыром                  ?ккал  🟢 рекомендовано
  [7] Сосиски вареные Сливочные                     ?ккал  🟢 рекомендовано


### 2.3 Добавить порцию в дневник

Укажи **индекс строки** из поиска выше и **граммы**.

In [5]:
# ↓↓↓ ВЫБЕРИ СТРОКУ И ГРАММЫ ↓↓↓
row_index = 0
grams = 60

item = to_fooditem(res.iloc[row_index])
log.add(item, grams)
p = item.portion(grams)
icon = LABEL_ICONS.get(res.iloc[row_index]['diabetes_label'], '?')
print(f'Добавлено: {item.name[:40]}, {grams} г → {p["kcal"]} ккал (Б{p["protein_g"]} Ж{p["fat_g"]} У{p["carbs_g"]})')
print(f'Метка для диабета: {icon}')
print(f'  {res.iloc[row_index]["diabetes_reason"]}')

Добавлено: Гречка, 60 г → 201.6 ккал (Б7.68 Ж1.92 У38.4)
Метка для диабета: ✅ разрешено
  ГИ 48 ≤55 — низкий, разрешено


### 2.4 Текущее состояние дня

Заполни норму, повторяя шаги 2.2–2.3. Здесь — сводка и предупреждения.

In [6]:
print('Съедено за день:')
print(log.to_df().to_string(index=False))
print()
print('Выполнение цели (КБЖУ):')
print(log.compare(target).to_string(index=False))
print()
day = log.totals
checks = check_day(day, 'diabetes_t2', target_kcal=target.target_kcal)
if checks:
    print('⚠️ Предупреждения по диабету:')
    for c in checks:
        print(f'  {c}')
else:
    print('✅ Лимиты сахара и натрия — в пределах нормы.')

Съедено за день:
Продукт Источник  Граммы  ккал  Б (г)  Ж (г)  У (г)
 Гречка      off      60 201.6   7.68   1.92   38.4

Выполнение цели (КБЖУ):
      Нутриент  Факт  Цель  Осталось  % вып.
Калории (ккал)   202  1614      1412      12
     Белки (г)     8    73        65      11
      Жиры (г)     2    54        52       4
  Углеводы (г)    38   210       171      18

✅ Лимиты сахара и натрия — в пределах нормы.


## Управление

- Отменить последнюю запись: `log.pop()`
- Очистить день: `log.clear()`

In [7]:
# log.pop()    # раскомментируй, чтобы убрать последнюю порцию
print(f'Записей в дневнике: {len(log.to_df())}')

Записей в дневнике: 1
